In [ ]:
# ============================================
# SESSION 4 - LOGISTIC REGRESSION & PIPELINE
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================
# 1. LOAD DATASET
# ============================================

CSV_PATH = "/content/placement_predict_50k Dataset.csv"

df = pd.read_csv(
    CSV_PATH,
    on_bad_lines="skip",
    engine="python"
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================
# 2. BASIC CLEANING
# ============================================

# Remove completely empty rows
df = df.dropna(how="all")

print("\nShape after cleaning:", df.shape)


# ============================================
# 3. CREATE CGPA_Tier
# ============================================

# Check if CGPA column exists
if "CGPA" in df.columns:

    def create_cgpa_tier(cgpa):
        if pd.isna(cgpa):
            return np.nan
        elif cgpa < 6:
            return "Low"
        elif cgpa < 8:
            return "Medium"
        else:
            return "High"

    df["CGPA_Tier"] = df["CGPA"].apply(create_cgpa_tier)

    print("\nCGPA_Tier created successfully!")
    print(df["CGPA_Tier"].value_counts(dropna=False))

else:
    print("\nCGPA column not found.")


# ============================================
# 4. BINARY LOGISTIC REGRESSION
#    TARGET = PlacementStatus
# ============================================

target = "PlacementStatus"

# Remove rows where target is missing
data_binary = df.dropna(subset=[target]).copy()

X = data_binary.drop(columns=[target])
y = data_binary[target]

print("\nBinary target:", target)
print("Classes:")
print(y.value_counts())


# ============================================
# 5. IDENTIFY NUMERICAL AND CATEGORICAL COLUMNS
# ============================================

numeric_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nNumerical columns:")
print(numeric_features)

print("\nCategorical columns:")
print(categorical_features)


# ============================================
# 6. PREPROCESSING PIPELINE
#    Numerical -> Impute + Scale
#    Categorical -> Impute + OHE
# ============================================

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)


# ============================================
# 7. TRAIN-VALIDATION SPLIT
# ============================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])


# ============================================
# 8. LOGISTIC REGRESSION PIPELINE
# ============================================

lr_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000
        ))
    ]
)


# ============================================
# 9. TRAIN MODEL
# ============================================

lr_model.fit(X_train, y_train)

print("\nLogistic Regression trained successfully!")


# ============================================
# 10. PREDICTIONS
# ============================================

y_train_pred = lr_model.predict(X_train)
y_val_pred = lr_model.predict(X_val)


# ============================================
# 11. ACCURACY
# ============================================

train_accuracy = accuracy_score(
    y_train,
    y_train_pred
)

val_accuracy = accuracy_score(
    y_val,
    y_val_pred
)

print("\n========== LOGISTIC REGRESSION ==========")

print("Training Accuracy:",
      round(train_accuracy, 4))

print("Validation Accuracy:",
      round(val_accuracy, 4))


# ============================================
# 12. CLASSIFICATION REPORT
# ============================================

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_val_pred
    )
)


# ============================================
# 13. CONFUSION MATRIX
# ============================================

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val,
        y_val_pred
    )
)

Dataset loaded successfully!
Shape: (38496, 32)

Columns:
['StudentID', 'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'CGPA_Tier', 'PlacementStatus', 'IsAnomaly', 'Salary Package']

Shape after cleaning: (38496, 32)

CGPA_Tier created successfully!
CGPA_Tier
Medium    16057
High      13155
Low        9283
NaN           1
Name: count, dtype: int64

Binary target: PlacementStatus
Classes:
PlacementStatus
1.0    25307
0.0    13188
Name: count, dtype: int64

Numerical columns:
['StudentID', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Proje